> 📅 __Date: 2026-08-20__

# 🧠 **Advanced Attention & LLM Training**

> **Goal:** Understand how modern attention implementations reduce inference cost and memory usage, and then understand how an LLM moves from raw pre-training to instruction following and human-aligned behavior.

---

# ⏱️ **Autoregressive Decoding and Time Steps**

A decoder-only LLM generates text **one token at a time**.

At each time step, the tokens already generated become the input context for predicting the next token.

| Time step | Input to decoder | Output of decoder |
|---|---|---|
| **t=0** | `<SoS>` | `Hum` |
| **t=1** | `<SoS> Hum` | `transformer` |
| **t=2** | `<SoS> Hum transformer` | `sikh` |
| **t=3** | `<SoS> Hum transformer sikh` | `rahe` |
| **t=4** | `<SoS> Hum transformer sikh rahe` | `hai` |
| **t=5** | `<SoS> Hum transformer sikh rahe hai` | `<EoS>` |

**Conceptually:**

```text
<SoS>
  ↓
Hum
  ↓
transformer
  ↓
sikh
  ↓
rahe
  ↓
hai
  ↓
<EoS>
```

The important point is that the model does **not** normally generate all future tokens in one autoregressive step. Each newly generated token becomes part of the context for the next prediction.

---

# ⚡ **To Reduce Latency**

**During autoregressive generation, there is a major inefficiency:**

```text
At every new token:
Recompute information about tokens that were already processed
```

This motivates the **KV Cache**.

---

# 💾 **KV Cache**

## **KV Cache = Stored previously calculated Key & Value vectors to use in the next time steps**

**In self-attention, each token produces:**

```text
Query (Q)
Key   (K)
Value (V)
```

The query for the current generation step needs to interact with keys and values from previous positions.

Without caching, the model would repeatedly calculate K and V information for earlier tokens.

**With KV Cache:**

```text
Previous tokens
      ↓
Previously calculated K & V
      ↓
       KV Cache
      ↓
Reuse in the next time step
```

### **Why does this reduce latency?**

Suppose the model has already generated many tokens.

At the next step, the **old Key and Value vectors are already known**. There is no reason to calculate them again just to use them for the current query.

**So:**

```text
Without KV Cache
→ Recalculate previous K/V information

With KV Cache
→ Reuse previous K/V information
```

> **KV Cache trades memory for faster autoregressive inference.**

---

# 📦 **KV Cache Size**

The KV Cache grows with the number of previously processed tokens.

A simplified way to count stored K/V vectors is:

$$
\text{KV vectors}
=
\text{Tokens}
\times
2
\times
\text{KV heads}
$$

**where:**

```text
2 → Key + Value
```

## **Example: 100 Tokens**

**Your simplified example:**

```text
heads = 6

100 × 2 × 6
```

**Therefore:**

$$
100 \times 2 \times 6 = 1200
$$

This gives the **number of K/V vectors** in the simplified counting model.

> **Important:** This is not yet the actual memory in bytes. To calculate memory usage, we also need the vector dimensionality (`head_dim`) and the number of bytes used per element (for example, FP16/BF16/FP8).

**A more complete conceptual expression is:**

$$
\text{KV memory}
\propto
\text{tokens}
\times
\text{KV heads}
\times
2
\times
\text{head dimension}
\times
\text{bytes per element}
$$

---

# ⚠️ **Why Does KV Cache Require More Storage?**

Because the cache stores information for **all previously processed tokens**, its memory requirement grows as the sequence becomes longer.

**For example:**

```text
10 tokens
→ store K/V for 10 positions

100 tokens
→ store K/V for 100 positions

10,000 tokens
→ store K/V for 10,000 positions
```

**So:**

> **Longer context → larger KV Cache → more memory required**

This is one of the reasons modern LLM architectures use methods such as **MQA** and **GQA**.

---

# 🧩 **MHA vs MQA vs GQA**

<div align="center">

<img src="assets/MQA_GQA.png" width="800" alt="Multi-Head Attention, Multi-Query Attention, and Grouped-Query Attention">

<p><em>Figure: Comparison of Multi-Head Attention (MHA), Multi-Query Attention (MQA), and Grouped-Query Attention (GQA).</em></p>

</div>

The image illustrates how many K/V vectors need to be maintained for one token under different attention designs.

**According to your diagram:**

```text
MHA → 1 token = 16 Vectors

MQA → 1 token = 2 Vectors

GQA → 1 token = 8 Vectors
```

The exact counts above are tied to the configuration shown in the diagram.

---

# 🧠 **MHA — Multi-Head Attention**

> **MHA = Using different Key & Value vectors for each head.**

In standard multi-head attention, every attention head has its own K and V projections.

**Conceptually:**

```text
Head 1 → K₁, V₁
Head 2 → K₂, V₂
Head 3 → K₃, V₃
...
```

So if there are many attention heads, there are also many K/V vectors to store.

### **Advantage**

Different heads can independently learn different attention patterns.

### **Disadvantage**

More K/V heads mean a **larger KV Cache**.

---

# 🧠 **MQA — Multi-Query Attention**

> **MQA = Every head will use the same Key & Value.**

**In MQA:**

```text
Head 1 ─┐
Head 2 ─┤
Head 3 ─┤──→ Same K and V
Head 4 ─┤
...     ─┘
```

The Query vectors can remain separate for different heads, but the Key and Value representations are shared.

**Therefore:**

```text
Many Query heads
        +
Shared K/V
        ↓
Much smaller KV Cache
```

### **Main benefit**

> **MQA dramatically reduces the amount of K/V state that must be stored and moved during autoregressive decoding.**

### **Trade-off**

The shared K/V representation reduces the amount of independent K/V information available to different query heads, which can hurt quality in some settings.

### **Example**

```text
MQA → Falcon
```

---

# 🧠 **GQA — Grouped-Query Attention**

> **GQA = Each group will use the same Key & Value.**

GQA is a middle ground between MHA and MQA.

**Instead of:**

```text
Every head has its own K/V
```

**or:**

```text
All heads share one K/V
```

**we group several query heads together:**

```text
Query Head 1 ─┐
Query Head 2 ─┤ → K/V Group 1
Query Head 3 ─┘

Query Head 4 ─┐
Query Head 5 ─┤ → K/V Group 2
Query Head 6 ─┘
```

**Thus:**

```text
MHA → one K/V set per head

GQA → one K/V set per group

MQA → one shared K/V set
```

GQA therefore provides a useful **quality vs memory** compromise.

### **Example**

```text
GQA → Llama series
```

---

# 📊 **Accuracy**

**Your notes summarize the usual trade-off as:**

$$
\text{MHA} > \text{GQA} > \text{MQA}
$$

**This means:**

```text
MHA
→ Highest attention-head independence

GQA
→ Middle ground

MQA
→ Strongest K/V sharing
```

The exact quality difference depends on the model architecture and training configuration.

---

# 💾 **Space Optimization**

**Your notes summarize the memory-efficiency ordering as:**

$$
\text{MQA} > \text{GQA} > \text{MHA}
$$

**Meaning:**

```text
MQA
→ Most KV-cache efficient

GQA
→ Middle ground

MHA
→ Largest KV-cache requirement
```

> **Memory trick:**  
> **More K/V sharing → less KV Cache memory.**

---

# 🪟 **Sliding Window Attention**

```text
Sliding Window Attention → Mistral
```

<div align="center">

<img src="assets/Sliding-window-attention.png" width="800" alt="Vanilla Attention, Sliding Window Attention, and Effective Context Length">

<p><em>Figure: Vanilla attention, sliding-window attention, and effective context length.</em></p>

</div>

---

# 🌐 **Vanilla Attention**

> **Vanilla Attention = Calculate attention with all past tokens.**

For a causal decoder, the token at position \(t\) can attend to earlier positions.

**Conceptually:**

```text
Token 1 → 1

Token 2 → 1, 2

Token 3 → 1, 2, 3

Token 4 → 1, 2, 3, 4

...
```

As the sequence grows, the number of positions available to attend to grows as well.

---

# 🪟 **Sliding Window Attention**

> **Sliding Window Attention = During attention calculation, only past tokens within the window range are considered.**

Instead of allowing attention to all earlier tokens, we define a fixed window.

**Your example:**

```text
I born in Germany .... so I speak --?
```

**Suppose:**

```text
1 2 3 .. 10
```

**With a window of 3:**

```text
1 → 1

2 → 1, 2

3 → 1, 2, 3

4 → 2, 3, 4

...
```

The window **slides forward** as the sequence grows.

### **Why is this useful?**

**Without a window:**

```text
Current token
     ↓
All previous tokens
```

With a sliding window:

```text
Current token
     ↓
Only nearby tokens inside the window
```

This limits the amount of context directly considered by each attention operation and can reduce the computational/memory burden associated with very long sequences.

### ⚠️ **Important Intuition**

Sliding-window attention does **not** necessarily mean the model has absolutely no information about distant text.

Information can propagate across multiple layers and positions over time.

**So the key idea is:**

> **Direct attention is local, while information can still move through the network indirectly.**

---

# ⚡ **Flash Attention**

<div align="center">

<img src="assets/flash-attention.png" width="800" alt="Standard Attention Implementation vs Flash Attention Implementation">

<p><em>Figure: Standard attention implementation compared with Flash Attention.</em></p>

</div>

# **Flash Attention = It is optimized on memory level**

Flash Attention is best understood as an **IO-aware attention implementation**.

The mathematical attention operation is still the same idea, but the implementation is reorganized to reduce expensive memory traffic between GPU memory levels.

**Conceptually:**

```text
Standard implementation
→ Materialize large intermediate attention data
→ Move more data through memory
→ Higher memory traffic

Flash Attention
→ Process attention in memory-efficient tiles/blocks
→ Avoid unnecessary large intermediate materialization
→ Reduce memory traffic
```

**So the key point is:**

> **Flash Attention is primarily an implementation-level optimization, not a new attention formula.**

**This distinction is important:**

```text
MHA / MQA / GQA
→ Change how attention heads share K/V

Sliding Window Attention
→ Changes which tokens can directly attend to which

Flash Attention
→ Changes how the same attention computation is implemented efficiently
```

---

# ⚡ **Gemini-flash**

> **Gemini Flash** belongs to a different category from **Flash Attention**.

**The word **Flash** can therefore be confusing:**

```text
Flash Attention
→ Attention implementation / kernel optimization

Gemini Flash
→ A model family designed for fast and efficient inference
```

So these should **not** be treated as the same technology.

---

# 🤖 **How LLMs Are Trained**

**Before discussing the training stages, it is useful to separate two ideas:**

```text
Training
→ Change the model parameters

Inference
→ Use the learned parameters to generate text
```

The training process is therefore about learning useful parameters, while decoding and generation happen after or during evaluation/inference.

---

# 🧰 **Hugging Face Example**

**A pretrained GPT-2 model can be loaded using:**

```python
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("gpt2")

model = AutoModelForCausalLM.from_pretrained("gpt2")
```

**Then:**

```python
prompt = "what is capital of India?"

input_ids = tokenizer(prompt, return_tensors="pt")

gen_tokens = model.generate(**input_ids)

tokenizer.decode(gen_tokens)
```

This example is performing **generation with an already-trained model**.

It is therefore primarily demonstrating **inference**, not the training loop itself.

---

# ⚠️ **Generation Warning**

**The shown output contains:**

```text
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=26) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
```

### **What does this mean?**

The generation call did not explicitly specify `max_new_tokens`, so the Transformers generation system used a default `max_length`.

**A clearer modern generation example is:**

```python
gen_tokens = model.generate(
    **input_ids,
    max_new_tokens=50
)
```

**The important distinction is:**

```text
max_new_tokens
→ How many NEW tokens can be generated

max_length
→ Total sequence length limit in the relevant generation setup
```

---

# 📝 **Example Output**

**The output shown in your notes is:**

```text
['what is capital of India?\n\nThe answer is that capital is the sum of all the things that are capitalised. Capital']
```

**This is a useful reminder that:**

> **A language model's output is generated from learned token probabilities; it is not guaranteed to be factually correct simply because the prompt asks a factual question.**

---

# 🏛️ **LLM — Large Language Model**

# **LLM = Large Language Model**

A language model learns a probability distribution over sequences of language.

The word **Large** generally refers to scale in areas such as:

```text
-- Number of parameters
-- Training data
```

The model can be considered large because it has a large number of learnable parameters and is trained on a large corpus.

---

# 🏋️ **How to Train an LLM?**

**Your notes organize modern LLM development into three major stages:**

```text
1. Pre-training : (Next Token Prediction)

2. Supervised Fine-Tuning (SFT)

3. Reinforcement Learning From Human Feedback (RLHF)
```

**A useful mental model is:**

```text
General language learning
        ↓
Instruction following
        ↓
Preference / behavior alignment
```

---

# 1️⃣ **Pre-training — Next Token Prediction**

## **What is pre-training?**

> **Pre-training = training the LLM to predict the next token from the previous context.**

**Your examples:**

```text
We
↓
are

we are
↓
training

we are training
↓
LLM
```

The same idea applies across a huge training corpus.

For each position, the model receives context and learns to assign high probability to the observed next token.

**Mathematically, for a token sequence:**

$$
x_1,x_2,\ldots,x_T
$$

**the autoregressive objective models:**

$$
P(x_t\mid x_{<t})
$$

where:

$$
x_{<t}=(x_1,x_2,\ldots,x_{t-1})
$$

**The training objective is commonly expressed as minimizing next-token prediction loss:**

$$
\mathcal{L}
=
-\sum_{t=1}^{T}
\log P(x_t\mid x_{<t})
$$

The model therefore receives a learning signal whenever its predicted distribution differs from the observed next token.

---

# 🧠 **What Does the Model Learn During Pre-training?**

**Your notes identify:**

```text
-- Language Understanding
-- Facts
```

These are important outcomes, but there is an even more useful way to think about it:

> **The model learns statistical structure in language and the information patterns present in its training data.**

**This can include:**

```text
Language patterns
Grammar
Common facts
World knowledge patterns
Code patterns
Style patterns
Reasoning-related patterns
```

**The exact capabilities depend strongly on:**

```text
Model architecture
+
Training data
+
Data quality
+
Training objective
+
Training scale
```

---

# 🧱 **Base / Foundation Model**

> **Base/Foundation Model = A pretrained model before instruction-specific alignment/fine-tuning stages.**

The pretrained model has learned broad language patterns, but it may not automatically behave like a helpful assistant.

**For example, a base model may be very good at predicting what text comes next without being specifically optimized to follow instructions such as:**

```text
"Summarize this paragraph."

"Write Python code."

"Explain this concept simply."
```

This is one reason SFT is introduced.

---

# 2️⃣ **Supervised Fine-Tuning (SFT)**

# **SFT = Supervised Fine-Tuning**

## **Supervised**

> **Supervised = training with labelled data.**

**In SFT, examples are deliberately provided as:**

```text
Input / Prompt
        ↓
Desired Response
```

The model is then trained to produce the target response.

---

# 📝 **SFT Example**

| Prompt (Input to LLM) | Response |
|---|---|
| **What is capital of India?** | **New Delhi** |
| **Define LLM** | **LLM is ...** |
| **Write a code ...** | `print("Hello World")` |

The important difference from raw pre-training is the **structure of the data**.

### **Pre-training**

```text
"We are"
→ predict "training"
```

### **SFT**

```text
Prompt:
"What is capital of India?"

Target response:
"New Delhi"
```

Thus SFT teaches the model to map prompts and instructions to desired answer formats.

---

# 🎯 **Why Do We Need SFT?**

A pretrained model learns:

> **What token is likely to come next?**

An instruction-tuned model is further trained to learn:

> **How should I respond to this instruction?**

So:

```text
Pre-training
→ General language capability

SFT
→ Instruction-following behavior
```

---

# 3️⃣ **Reinforcement Learning From Human Feedback (RLHF)**

# **RLHF = Reinforcement Learning From Human Feedback**

## 🎯 **Purpose of RLHF**

> **To align the model with human values.**

The central idea is that simply knowing language is not enough.

A model may generate several responses that are all grammatically valid, but humans may strongly prefer one over another.

**For example:**

```text
Response 1 → technically valid
Response 2 → more helpful
Response 3 → unsafe / irrelevant
```

RLHF introduces a mechanism for learning those preferences.

---

# 🏆 **Reinforcement Learning = Reward & Penalty**

In reinforcement learning, an action is evaluated using a reward signal.

**Conceptually:**

```text
Good behavior
→ higher reward

Undesired behavior
→ lower reward / penalty
```

The model can then be optimized so that actions associated with higher reward become more likely.

---

# 👤 **Data from Human Feedback**

**Your example:**

| Prompt | Response | Score |
|---|---|---:|
| **Prompt** | Response 1 | **50** |
| **Prompt** | Response 2 | **100** |
| **Prompt** | Response 3 | **0** |

The idea is that humans provide preference information about candidate responses.

**In a real preference-learning pipeline, the feedback may be represented as:**

```text
Prompt
   ↓
Several candidate responses
   ↓
Human preference / ranking
   ↓
Training signal
```

The exact scoring mechanism can vary across RLHF implementations.

---

# 🏆 **Reward Model**

> **Reward Model = To give points to the response generated by the model.**

The Reward Model learns to predict which responses humans are more likely to prefer.

**Conceptually:**

```text
Prompt + Response
        ↓
   Reward Model
        ↓
     Reward
```

**For example:**

```text
Response 1 → 50
Response 2 → 100
Response 3 → 0
```

The reward model is therefore an intermediate component that converts human preference patterns into a numerical signal.

---

# 🔄 **Reinforcement Learning**

**Your simplified pipeline:**

```text
Prompt
  ↓
LLM
  ↓
Response
  ↓
Reward Model
  ↓
Points
```

**A more complete conceptual loop is:**

```text
Prompt
  ↓
LLM / Policy
  ↓
Generate Response
  ↓
Reward Model
  ↓
Reward Signal
  ↓
Update Policy
  ↓
Generate better responses
  ↓
Repeat
```

This is the key reinforcement-learning idea:

> **The model is optimized not simply to imitate text, but to increase the expected reward produced by the learned preference signal.**

---

# 🖼️ **LLM Training Overview**

<div align="center">

<img src="assets/LLM_Training.png" width="800" alt="LLM Training: Low-Quality Data, High-Quality Data, Human Feedback, and RLHF">

<p><em>Figure: LLM training progression from low-quality data and high-quality data to human feedback and RLHF.</em></p>

</div>

The visual in your notes shows the progression from data quality toward human feedback and RLHF.

**Conceptually:**

```text
Low-quality data
        ↓
High-quality data
        ↓
Human Feedback
        ↓
RLHF
```

The important lesson is that **data quality and feedback quality matter throughout the LLM development pipeline**.

---

# 🧭 **Complete LLM Training Mental Model**

```text
                   Raw / Large-Scale Data
                           ↓
                    Pre-training
                           ↓
              Next Token Prediction
                           ↓
                Base / Foundation Model
                           ↓
                    SFT Data
                           ↓
             Supervised Fine-Tuning
                           ↓
              Instruction-Following Model
                           ↓
                Human Preference Data
                           ↓
                    Reward Model
                           ↓
                         RLHF
                           ↓
                 More Aligned Behavior
```

---

# 🆚 **Pre-training vs SFT vs RLHF**

| Stage | Main Input | Main Goal | Learning Signal |
|---|---|---|---|
| **Pre-training** | Large text/code corpus | Learn general language patterns | Next-token prediction |
| **SFT** | Labelled prompt-response pairs | Learn instruction following | Target response |
| **RLHF** | Human preference information | Align behavior with preferences | Reward |

### **Memory Trick**

```text
Pre-training
→ Learn language

SFT
→ Learn instructions

RLHF
→ Learn preferences / alignment
```

---

# 🔗 **How Advanced Attention Fits Into LLMs**

The attention optimizations and training stages solve **different problems**.

```text
Training stages
→ How the model learns

Attention optimizations
→ How the model computes attention efficiently

KV Cache
→ How inference reuses previous K/V

MHA / MQA / GQA
→ How K/V are shared across heads

Sliding Window Attention
→ How much context participates directly in attention

Flash Attention
→ How attention is implemented efficiently in memory
```

This distinction is extremely useful.

> **Training tells the model what to learn. Attention optimizations help the model perform that computation efficiently.**

---

# 📌 **Quick Revision**

```text
Autoregressive Generation
→ Generate one token at a time using previous context

KV Cache
→ Store previously calculated Key & Value vectors for reuse

KV Cache Benefit
→ Reduce repeated K/V computation during generation

KV Cache Cost
→ Requires additional memory

MHA
→ Different Key & Value vectors for each head

MQA
→ Every head shares the same Key & Value

GQA
→ Each group of heads shares the same Key & Value

Accuracy
→ MHA > GQA > MQA

Space Optimization
→ MQA > GQA > MHA

Sliding Window Attention
→ Attend only to tokens inside a local window

Flash Attention
→ IO-aware / memory-efficient implementation of attention

Gemini Flash
→ A model family, not the same thing as Flash Attention

LLM
→ Large Language Model

Why large?
→ Number of parameters + training data

Pre-training
→ Next Token Prediction

Pre-training learns
→ Language understanding + information patterns / facts

Base/Foundation Model
→ Pretrained model before instruction-specific tuning

SFT
→ Supervised Fine-Tuning with labelled prompt-response data

SFT purpose
→ Improve instruction following

RLHF
→ Reinforcement Learning From Human Feedback

RLHF purpose
→ Align behavior with human preferences / values

Reward Model
→ Assigns a reward score to model responses

Reinforcement Learning
→ Uses reward signals to optimize model behavior
```

---

# 🧠 **Final Mental Model**

```text
                    ADVANCED ATTENTION
                           │
            ┌──────────────┼───────────────┐
            ↓              ↓               ↓
        KV Cache         MHA/GQA/MQA    Sliding Window
            │              │               │
            ↓              ↓               ↓
       Reuse K/V       Reduce K/V       Local Context
            │          memory cost           │
            └──────────────┬────────────────┘
                           ↓
                   Efficient Attention
                           │
                           ↓
                    Flash Attention
                           │
                           ↓
                 Efficient Computation


                      LLM TRAINING
                           │
                           ↓
                    Large-Scale Data
                           │
                           ↓
                     Pre-training
                           │
                           ↓
               Next Token Prediction
                           │
                           ↓
                Base/Foundation Model
                           │
                           ↓
                      SFT
                           │
                           ↓
              Instruction-Following Model
                           │
                           ↓
                  Human Preference Data
                           │
                           ↓
                    Reward Model
                           │
                           ↓
                        RLHF
                           │
                           ↓
                 Aligned Model Behavior
```

---

# 📚 **Key Takeaways**

> **1. KV Cache reduces inference latency by reusing previously computed Key and Value vectors, at the cost of additional memory.**

> **2. MQA and GQA reduce KV Cache size by sharing Key/Value representations across multiple query heads.**

> **3. Sliding Window Attention reduces the amount of context directly attended to at each position.**

> **4. Flash Attention is an implementation-level, IO-aware optimization that makes attention more memory-efficient.**

> **5. Pre-training teaches a model general language patterns through next-token prediction.**

> **6. SFT teaches the pretrained model how to follow labelled instructions.**

> **7. RLHF uses human preference signals to push the model toward more desirable behavior.**

> **8. The full picture is: learn general language → learn instructions → optimize behavior, while efficient attention techniques make the resulting model practical to run at scale.**
